In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd

import model.assumptions as assumptions_module
import model.policy as policy_module
import model.valuation as valuation_module
import model.scenarios as scenarios_module
import model.capital as capital_module

from model.capital.correlation_loader import (
    load_correlation_matrix
)

from model.capital import (
    aggregate_life_scr,
    aggregate_market_scr,
    aggregate_basic_scr
)

from model.scenarios.scenario_loader import (
    load_scenarios
)

In [ ]:
mortality_table = (
    assumptions_module.load_mortality_table(
        "../data/mortality_tables/ons_mortality.csv"
    )
)

mortality_parameters = (
    assumptions_module.load_mortality_parameters(
        "../data/mortality_parameters/smoker_multipliers.csv"
    )
)

mortality_table.mortality_parameters = (
    mortality_parameters
)

yield_curve = (
    assumptions_module.load_yield_curve(
        "../data/yield_curves/sonia_spot_rates.csv"
    )
)

lapse_table = (
    assumptions_module.load_lapse_table(
        "../data/lapse_tables/lapse_rates.csv"
    )
)

expense_table = (
    assumptions_module.load_expense_table(
        "../data/expense_tables/expense_rates.csv"
    )
)

assumptions = assumptions_module.AssumptionSet(
    providers={
        "mortality": mortality_table,
        "interest": yield_curve,
        "lapse": lapse_table,
        "expenses": expense_table
    }
)

In [3]:
scenarios = load_scenarios(
    "../data/scenarios/standard_formula.csv"
)

In [4]:
life_matrix = load_correlation_matrix(
    "../data/correlations/life/life_correlation_matrix.csv"
)

market_matrix = load_correlation_matrix(
    "../data/correlations/market/market_correlation_matrix.csv"
)

bscr_matrix = load_correlation_matrix(
    "../data/correlations/bscr/bscr_correlation_matrix.csv"
)

In [5]:
policy = policy_module.Policy(
    age=40,
    term=20,
    sum_assured=100000,
    premium=500,
    gender="M",
    smoker_status="Non-Smoker",
    product_type="Term"
)

In [6]:
base_result = valuation_module.value_policy(
    policy,
    assumptions,
    return_breakdown=True
)

print("Base BEL:")
print(base_result.best_estimate_liability)

print("\nBase PVFP:")
print(base_result.pv_future_profit)

AttributeError: 'NoneType' object has no attribute 'expense'

In [ ]:
enabled_scenarios = [
    scenario
    for scenario in scenarios
    if scenario.enabled
]

In [ ]:
scr_results = []

for scenario in enabled_scenarios:

    stressed_result = (
        scenarios_module.run_scenario(
            policy,
            assumptions,
            scenario
        )
    )

    scr = capital_module.calculate_scr(
        base_result,
        stressed_result,
        scenario
    )

    scr_results.append(scr)

In [ ]:
life_scrs = [
    scr
    for scr in scr_results
    if scr.aggregation_category
    == "life"
]

market_scrs = [
    scr
    for scr in scr_results
    if scr.aggregation_category
    == "market"
]

In [ ]:
life_scr = aggregate_life_scr(
    life_scrs,
    life_matrix
)

market_scr = aggregate_market_scr(
    market_scrs,
    market_matrix
)

bscr = aggregate_basic_scr(
    life_scr,
    market_scr,
    bscr_matrix
)

In [ ]:
print("Life SCR:")
print(life_scr.to_dict())

print("\nMarket SCR:")
print(market_scr.to_dict())

print("\nBSCR:")
print(bscr.to_dict())